In [ ]:
import torch
import hydra
import train
import pathlib
import src.utils.rebasin.activation_matching
import numpy as np
import src.utils.interpolation
from .checkpoint_directories import checkpoint_directories_by_architecture

device="cuda:0"

checkpoint_directories = checkpoint_directories_by_architecture["mlp"]
checkpoint_path_1, checkpoint_path_2 = [checkpoint_directories["mlp_symmetry0"]+f"/checkpoint_epoch_100_model_{i}.pt" for i in [3, 4]]

with hydra.initialize(version_base=None, config_path=str(pathlib.Path(checkpoint_path_1).parent)):
    cfg = hydra.compose(config_name="config")
cfg.dataset.batch_size = 200
data_info = train.setup_data_loaders(cfg)

model1 = train.create_model(cfg, mask_seed=-1).to(device)
model1_permuted = train.create_model(cfg, mask_seed=-1).to(device)
model2 = train.create_model(cfg, mask_seed=-1).to(device)
model2_permuted = train.create_model(cfg, mask_seed=-1).to(device)
model1.load_state_dict(torch.load(checkpoint_path_1, map_location=device)["model_state_dict"])
model2.load_state_dict(torch.load(checkpoint_path_2, map_location=device)["model_state_dict"])

mini_train_loader = [next(iter(data_info["train_loader"]))]

In [14]:
permutation_spec = src.utils.rebasin.mlp_permutation_spec(3, norm=True, bias=True)
output_permutation_by_layer = src.utils.rebasin.activation_matching.activation_matching(
    permutation_spec, model1, model2, mini_train_loader, device=device
)
output_permutation_by_layer[("post_linear_activation_matching_mode", src.utils.rebasin.ActivationCorrelationMode.PearsonCorrelation)]["lins.0"].activation_similarities.max()

tensor(0.9525, device='cuda:0')

In [2]:
import copy

mini_train_loader = [next(iter(data_info["train_loader"]))]
permutation_spec = src.utils.rebasin.activation_matching.mlp_permutation_spec(3, norm=True, bias=True)
output_permutation_by_layer = src.utils.rebasin.activation_matching.activation_matching(
    permutation_spec, model1, model2, data_info["train_loader"], device=device
)
permuted_params_1, permuted_params_2 = [src.utils.rebasin.activation_matching.apply_permutation(
    permutation_spec,
    {"P_0": output_permutation_by_layer["lins.1.input"].optimal_permutation,
     "P_1": output_permutation_by_layer["lins.2.input"].optimal_permutation,
     "P_2": output_permutation_by_layer["lins.3.input"].optimal_permutation},
    copy.deepcopy(m).state_dict()
) for m in [model1, model2]]
model1_permuted.load_state_dict(permuted_params_1)
model2_permuted.load_state_dict(permuted_params_2)
output_permutation_by_layer_second_round = src.utils.rebasin.activation_matching.activation_matching(
    permutation_spec, model1, model2_permuted, data_info["train_loader"], device=device
)
print("Sanity check: this permutation should be 0, 1, 2, ..., 19")
print(output_permutation_by_layer_second_round["lins.1.input"].optimal_permutation[:20].tolist())

with torch.no_grad():
    print("Model 1 original <-> model 2 original")
    src.utils.interpolation.interpolate_models(
        copy.deepcopy(model1), model1.state_dict(), model2.state_dict(),
        mini_train_loader, mini_train_loader, mini_train_loader, steps=4, device="cuda:0"); print();
    print("Model 1 original <-> model 2 permuted")
    src.utils.interpolation.interpolate_models(
        copy.deepcopy(model1), model1.state_dict(), permuted_params_2,
        mini_train_loader, mini_train_loader, mini_train_loader, steps=4, device="cuda:0"); print();

Sanity check: this permutation should be 0, 1, 2, ..., 19
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Model 1 original <-> model 2 original
Step  1/5 (λ=0.000): Train Acc=100.00%, Val Acc=100.00%, Test Acc=100.00%, Train Loss=0.0004, Val Loss=0.0004, Test Loss=0.0004
Step  2/5 (λ=0.250): Train Acc=100.00%, Val Acc=100.00%, Test Acc=100.00%, Train Loss=0.0076, Val Loss=0.0076, Test Loss=0.0076
Step  3/5 (λ=0.500): Train Acc=91.50%, Val Acc=91.50%, Test Acc=91.50%, Train Loss=0.3009, Val Loss=0.3009, Test Loss=0.3009
Step  4/5 (λ=0.750): Train Acc=99.50%, Val Acc=99.50%, Test Acc=99.50%, Train Loss=0.0085, Val Loss=0.0085, Test Loss=0.0085
Step  5/5 (λ=1.000): Train Acc=100.00%, Val Acc=100.00%, Test Acc=100.00%, Train Loss=0.0000, Val Loss=0.0000, Test Loss=0.0000

Model 1 original <-> model 2 permuted
Step  1/5 (λ=0.000): Train Acc=100.00%, Val Acc=100.00%, Test Acc=100.00%, Train Loss=0.0004, Val Loss=0.0004, Test Loss=0.0004
Step  2/5 (λ=0.250): Train Acc=1

In [3]:
random_permutations_by_layer = {
    key: torch.randperm(val.optimal_permutation.shape[0], device=val.optimal_permutation.device)
    for key, val in output_permutation_by_layer.items()
}
randomly_permuted_params_1, randomly_permuted_params_2 = [src.utils.rebasin.activation_matching.apply_permutation(
    permutation_spec,
    {"P_0": random_permutations_by_layer["lins.1.input"],
     "P_1": random_permutations_by_layer["lins.2.input"],
     "P_2": random_permutations_by_layer["lins.3.input"]},
    copy.deepcopy(m).state_dict()
) for m in [model1, model2]]

In [6]:
import torch
import torch.nn as nn
from torch.func import functional_call, jvp, vjp

# General version with proper Hessian handling
def ggn_quadratic_form_general(model, loss_fn, params_ref, params_1, params_2, inputs, targets):
    v = {k: params_1[k] - params_2[k] for k in params_1.keys()}
    # print({k: val.shape for k,val in v.items()})
    
    # Step 1: Compute Jv using forward-mode
    def residual_fn(p):
        return functional_call(model, p, inputs)
    
    residuals_ref, jv = jvp(residual_fn, (params_ref,), (v,))
    # print("jv:", jv.shape)
    # print("residuals_ref:", residuals_ref.shape)
    
    # # Step 2: Compute Hessian-vector product using backward-mode
    # # Make residuals require grad for Hessian computation
    # residuals_ref_grad = residuals_ref.detach().requires_grad_(True)
    # loss_val = loss_fn(residuals_ref_grad, targets)
    # print("residuals_ref_grad:", residuals_ref_grad.shape)
    # print("loss_val:", loss_val.shape)
    
    # # Compute gradient of loss w.r.t. residuals
    # grad_r, = torch.autograd.grad(loss_val, residuals_ref_grad, create_graph=True)
    # print("grad_r:", grad_r.shape)
    
    # # Compute HVP: gradient of (grad_r · jv) w.r.t. residuals
    # hvp, = torch.autograd.grad(
    #     grad_r, residuals_ref_grad, grad_outputs=jv,
    #     retain_graph=False
    # )
    
    
    _, hvp_alternative = torch.autograd.functional.hvp(
        func=lambda f_of_theta: loss_fn(f_of_theta, targets),
        inputs=residuals_ref.detach(),
        v=jv
    )
    # print(hvp)
    # print(hvp_alternative)
    # print(torch.isclose(hvp, hvp_alternative).float().mean())

    # Step 3: Compute (Jv)ᵀ H (Jv)
    # print("hvp:", hvp.shape)
    return torch.sum(jv * hvp_alternative, dim=1)


{'lins.0.weight': torch.Size([512, 784]), 'lins.0.bias': torch.Size([512]), 'lins.1.weight': torch.Size([512, 512]), 'lins.1.bias': torch.Size([512]), 'lins.2.weight': torch.Size([512, 512]), 'lins.2.bias': torch.Size([512]), 'lins.3.weight': torch.Size([10, 512]), 'lins.3.bias': torch.Size([10]), 'norms.0.weight': torch.Size([512]), 'norms.0.bias': torch.Size([512]), 'norms.1.weight': torch.Size([512]), 'norms.1.bias': torch.Size([512]), 'norms.2.weight': torch.Size([512]), 'norms.2.bias': torch.Size([512])}
jv: torch.Size([200, 10])
residuals_ref: torch.Size([200, 10])


7.394721615128219e-05

In [3]:
inputs = []
targets = []
for i, (input, target) in enumerate(data_info["train_loader"]):
    inputs.append(input)
    targets.append(target)
    if i >= 30: break

input0 = torch.cat(inputs).to(device)
target0 = torch.cat(targets).to(device)


# input0, target0 = next(iter(data_info["train_loader"])); input0 = input0.to(device); target0 = target0.to(device)

In [7]:
params_mid_aligned = {k: (dict(model1_permuted.named_parameters())[k]*0.5 + dict(model2.named_parameters())[k]*0.5).detach().clone() for k, v in model1.named_parameters()}
params_mid_unaligned = {k: (dict(model1.named_parameters())[k]*0.5 + dict(model2.named_parameters())[k]*0.5).detach().clone() for k, v in model1.named_parameters()}
params_1 = {k: v.detach().clone() 
            for k, v in model1.named_parameters()}
params_2 = {k: v.detach().clone() 
            for k, v in model2.named_parameters()}
params_2_permuted = {k: v.detach().clone()
            for k, v in model2_permuted.named_parameters()}

print("thetaA = params_1, thetaB = params_2, thetaRef = params_1")
print(ggn_quadratic_form_general(model1, torch.nn.CrossEntropyLoss(),
                           params_ref=params_1,
                           params_1=params_1,
                           params_2=params_2,
                           inputs=input0, targets=target0).mean().item())
print("thetaA = params_1, thetaB = params_2, thetaRef = params_2")
print(ggn_quadratic_form_general(model1, torch.nn.CrossEntropyLoss(),
                           params_ref=params_2,
                           params_1=params_1,
                           params_2=params_2,
                           inputs=input0, targets=target0).mean().item())
print("thetaA = params_1, thetaB = params_2, thetaRef = params_mid_unaligned")
print(ggn_quadratic_form_general(model1, torch.nn.CrossEntropyLoss(),
                           params_ref=params_mid_unaligned,
                           params_1=params_1,
                           params_2=params_2,
                           inputs=input0, targets=target0).mean().item())

thetaA = params_1, thetaB = params_2, thetaRef = params_1
1.655410414969083e-05
thetaA = params_1, thetaB = params_2, thetaRef = params_2
5.9145708775076855e-08
thetaA = params_1, thetaB = params_2, thetaRef = params_mid_unaligned
0.005990291479974985


In [8]:
print("thetaA = params_1, thetaB = params_2_permuted, thetaRef = params_1")
print(ggn_quadratic_form_general(model1, torch.nn.CrossEntropyLoss(),
                           params_ref=params_1,
                           params_1=params_1,
                           params_2=params_2_permuted,
                           inputs=input0, targets=target0).mean().item())
print("thetaA = params_1, thetaB = params_2_permuted, thetaRef = params_2_permuted")
print(ggn_quadratic_form_general(model1, torch.nn.CrossEntropyLoss(),
                           params_ref=params_2_permuted,
                           params_1=params_1,
                           params_2=params_2_permuted,
                           inputs=input0, targets=target0).mean().item())
print("thetaA = params_1, thetaB = params_2_permuted, thetaRef = params_mid_aligned")
print(ggn_quadratic_form_general(model1, torch.nn.CrossEntropyLoss(),
                           params_ref=params_mid_aligned,
                           params_1=params_1,
                           params_2=params_2_permuted,
                           inputs=input0, targets=target0).mean().item())

thetaA = params_1, thetaB = params_2_permuted, thetaRef = params_1
9.68075528362533e-06
thetaA = params_1, thetaB = params_2_permuted, thetaRef = params_2_permuted
0.014585400000214577
thetaA = params_1, thetaB = params_2_permuted, thetaRef = params_mid_aligned
6.53059032629244e-05
